In [ ]:
# import kagglehub
# kagglehub.login()  # Login to Kaggle using your credentials
# # Download latest version
# path = kagglehub.competition_download('rsna-knee-abnormality-detection', output_dir='/data/biophys/schimmenti/Repositories/camilla/misc/knee_data')
# 
# print("Path to competition files:", path)

In [ ]:
# import kagglehub
# path = kagglehub.model_download("metaresearch/dinov2/pyTorch/small", output_dir='/data/biophys/schimmenti/Repositories/camilla/misc/weights/dinov2_small')

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset
import pandas as pd
from pathlib import Path
from typing import Union, List
import pydicom as dicom
import matplotlib.pyplot as plt

class KneeDataset(Dataset):
    def __init__(self, root_dir: Union[str, Path], train: bool):
        self.root_dir = Path(root_dir) if isinstance(root_dir, str) else root_dir
        self.train = train
        self.series_df = pd.read_csv(self.root_dir / 'train_series.csv' if train else self.root_dir / 'test_series.csv')
        self.num_studies = self.series_df.groupby('StudyInstanceUID').ngroups
        self.data_dir = self.root_dir / 'train_series' if train else self.root_dir / 'test_series'
        self.annotations_df = pd.read_csv(self.root_dir / 'train.csv' if train else self.root_dir / 'test.csv')

    def __len__(self):
        return self.num_studies

    def __getitem__(self, idx):
        study_uid = self.series_df['StudyInstanceUID'].unique()[idx]
        series_uids = self.series_df[self.series_df['StudyInstanceUID'] == study_uid]['SeriesInstanceUID'].tolist()
        dirs =  {series_uid: self.data_dir / study_uid / series_uid for series_uid in series_uids}
        files = {series_uid: sorted(series_dir.glob('*.dcm')) for series_uid, series_dir in dirs.items()}
        loaded_files = {series_uid: { file.stem : dicom.dcmread(file).pixel_array for file in series_files} for series_uid, series_files in files.items()}
        return loaded_files

In [ ]:
dataset = KneeDataset('knee_data/', train=True)

In [ ]:
for series_uid, series_data in dataset[0].items():
    for instance_uid, image in series_data.items():
        print(f"Series UID: {series_uid}, Instance UID: {instance_uid}, Image shape: {image.shape}")

In [ ]:
from transformers import AutoModel

MODEL_PATH = "weights/dinov2_small/"  # folder containing config.json + model.safetensors

vision_model = AutoModel.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
)

vision_model.eval()

In [ ]:
import timm

vision_model   = timm.create_model(
    "convnextv2_tiny.fcmae",
    pretrained=True,
    num_classes=0,
    cache_dir='weights/convnextv2_tiny.fcmae/'
)


In [ ]:
class KneeVisionModel(nn.Module):
    def __init__(
        self,
        input_channels,
        kernel_size,
        stride,
        padding,
        backbone,
        backbone_input_channels,
        backbone_output_dim,
        num_classes,
        output_head_hidden_dims: Union[List[int], int] = 512,
        output_head_dropout=0.2,
    ):
        super().__init__()

        self.backbone = backbone
        self.input_channels = input_channels
        self.input_head = nn.Sequential(
            nn.Conv2d(
                input_channels,
                backbone_input_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding,
            ),
            nn.GELU(),
            nn.Conv2d(
                backbone_input_channels,
                backbone_input_channels,
                kernel_size=1,
                stride=1,
                padding=0,
            ),
        )

        self.backbone_output_dim = backbone_output_dim

        if isinstance(output_head_hidden_dims, int):
            output_head_hidden_dims = [output_head_hidden_dims]

        layers = [
            nn.LayerNorm(backbone_output_dim),
        ]
        prev_dim = backbone_output_dim
        for hidden_dim in output_head_hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(output_head_dropout),
            ])
            prev_dim = hidden_dim

        if len(output_head_hidden_dims) == 0:
            layers.append(nn.Dropout(output_head_dropout))

        layers.append(
            nn.Linear(prev_dim, num_classes)
        )

        self.head = nn.Sequential(*layers)

    def forward(self, x, slice_index):
        if x.ndim != 4:
            raise ValueError(f"Expected input tensor to have 4 dimensions (batch_size, channels, height, width), but got {x.ndim} dimensions.")
        B,C,H,W = x.shape
        if C != self.input_channels:
            raise ValueError(f"Expected input tensor to have {self.input_channels} channels, but got {C} channels.")
        if slice_index.ndim != 1 or slice_index.shape[0] != B:
            raise ValueError(f"Expected slice_index tensor to have shape (batch_size,), but got {slice_index.shape}.")

In [ ]:
knee_vision_model = KneeVisionModel(
    input_channels=1,
    kernel_size=3,
    stride=2,
    padding=3,
    backbone=vision_model,
    backbone_input_channels=3,
    backbone_output_dim=768,
    output_head_hidden_dims=[512],
    num_classes=12
)

In [ ]:
knee_vision_model(torch.as_tensor(image).unsqueeze(0).unsqueeze(0).float())